# Combined Loss Comparison for VessMAP Segmentation

This notebook demonstrates the new combined loss functionality in the TopologyLosses pipeline. We'll compare different combinations of segmentation and topological losses to find the optimal setup for vessel segmentation.

## Features Demonstrated:
- **Combined Loss Configuration**: Define weighted combinations of segmentation + topological losses
- **Automatic Metric Collection**: Segmentation metrics (Dice, IoU) + Topological metrics (Betti numbers, connectivity)
- **Training Comparison**: Compare performance across different loss combinations
- **Visualization**: Plot training curves and metric comparisons

## Section 1: Setup and Imports

Import required libraries and set up the environment.

In [ ]:
import gudhi
print(gudhi.__version__)


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import yaml
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict
import warnings
from torch.utils.tensorboard import SummaryWriter

In [ ]:
# Add project to path
sys.path.insert(0, '/Users/egor/VS_GIT_repositories/TopologyLosses')

# Import custom modules
from Train.pipeline import SegmentationPipeline, MultiLossComparator
from data.dataset import VessMapDataset, create_data_splits
from metrics.segmentation_metrics import SegmentationMetrics
from metrics.topological_metrcis import TopologicalMetrics
from losses.segmentation_losses.losses import get_segmentation_loss
from losses.topological_losses.losses import get_topological_loss

# Setup matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Suppress warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## Section 2: Define Combined Loss Configurations

Create different loss combinations to compare. Each configuration specifies weighted segmentation and topological losses.

In [ ]:
# Define different combined loss configurations to compare
loss_configs = {
    # 'baseline_dice': {
    #     'segmentation': [
    #         {'dice': {'weight': 1.0}}
    #     ]
    # },
    
    # 'dice_bce': {
    #     'segmentation': [
    #         {'dice': {'weight': 0.8}},
    #         {'bce': {'weight': 1.0}}
    #     ]
    # },
    
    # 'dice_topo': {
    #     'segmentation': [
    #         {'dice': {'weight': 0.8}}
    #     ],
    #     'topological': [
    #         {'betti': {'weight': 0.1}}
    #     ]
    # },
    
    # 'combined_topo': {
    #     'segmentation': [
    #         {'dice': {'weight': 0.5}},
    #         {'bce': {'weight': 0.5}}
    #     ],
    #     'topological': [
    #         {'bm': {'weight': 0.025}}
    #     ]
    # },

    'SAT_combined': {
        'segmentation': [
            {'dice': {'weight': 0.5}},
            {'bce': {'weight': 0.5}}
        ],
        'topological': [
            {'sat': {'weight': 0.05}}
        ]
    },
    
    # 'focal_topo': {
    #     'segmentation': [
    #         {'focal': {'weight': 1.0}}
    #     ],
    #     'topological': [
    #         {'hausdorff': {'weight': 0.01}}
    #     ]
    # }
}

print("Defined loss configurations:")
for name, config in loss_configs.items():
    print(f"\n{name}:")
    if 'segmentation' in config:
        seg_losses = [f"{list(entry.keys())[0]}({entry[list(entry.keys())[0]].get('weight', 1.0)})" 
                     for entry in config['segmentation']]
        print(f"  Segmentation: {', '.join(seg_losses)}")
    if 'topological' in config:
        topo_losses = [f"{list(entry.keys())[0]}({entry[list(entry.keys())[0]].get('weight', 1.0)})" 
                      for entry in config['topological']]
        print(f"  Topological: {', '.join(topo_losses)}")

## Section 3: Load Base Configuration

Load the base configuration and modify it for each loss setup.

In [ ]:
# Load base configuration
with open('configs/default_config.yaml', 'r') as f:
    base_config = yaml.safe_load(f)

print("Base configuration loaded:")
print(f"  Dataset: {base_config['data']['dataset_path']}")
print(f"  Model: {base_config['model']['name']}")
print(f"  Image size: {base_config['data']['image_size']}")
print(f"  Batch size: {base_config['data']['batch_size']}")
print(f"  Epochs: {base_config['training']['epochs']}")

# Create configurations for each loss setup
configs = {}
for loss_name, loss_config in loss_configs.items():
    config = base_config.copy()
    config['loss'] = loss_config
    # Reduce epochs for demo
    config['training']['epochs'] = 50
    configs[loss_name] = config

print(f"\nCreated {len(configs)} configurations for comparison")

## Section 4: Run Training Comparison

Train models with different loss combinations and collect results.

In [ ]:
# Initialize results storage
training_results = {}
validation_results = {}

print("Starting training comparison...")
print("=" * 80)

for loss_name, config in configs.items():
    print(f"\n{'='*80}")
    print(f"Training with {loss_name.upper()} configuration")
    print(f"{'='*80}")

    # Create pipeline without calling __init__ to avoid config loading
    pipeline = SegmentationPipeline('/Users/egor/VS_GIT_repositories/TopologyLosses/configs/default_config.yaml')
    
    # Train
    pipeline.train()
    
    # Store results
    training_results[loss_name] = {
        'train_loss': pipeline.history['train_loss'],
        'val_loss': pipeline.history['val_loss'],
        'train_metrics': dict(pipeline.history['train_metrics']),
        'val_metrics': dict(pipeline.history['val_metrics'])
    }
    
    # Evaluate on test set
    test_results = pipeline.evaluate(pipeline.test_loader, verbose=False)
    validation_results[loss_name] = test_results
    
    print(f"✓ {loss_name} training completed successfully")
    

print(f"\n{'='*80}")
print("TRAINING COMPARISON COMPLETED")
print(f"{'='*80}")
print(f"Successfully trained {len(training_results)}/{len(configs)} configurations")

In [ ]:
# Create pipeline without calling __init__ to avoid config loading
pipeline = SegmentationPipeline.__new__(SegmentationPipeline)
pipeline.config = config
pipeline.device = torch.device(config['training'].get('device', 'cpu'))
pipeline.epochs = config['training']['epochs']
pipeline._setup_directories()

# Initialize components
pipeline.model = pipeline._build_model()
pipeline.train_loader, pipeline.val_loader, pipeline.test_loader = pipeline._build_dataloaders()
pipeline.loss_fn = pipeline._build_loss()
pipeline.optimizer = pipeline._build_optimizer()
pipeline.scheduler = pipeline._build_scheduler()
pipeline.metrics = SegmentationMetrics()
pipeline.topological_metrics = TopologicalMetrics(threshold=config['training'].get('metric_threshold', 0.5))


In [ ]:
chpt = torch.load('/Users/egor/VS_GIT_repositories/TopologyLosses/checkpoints/best_model.pt', map_location=pipeline.device, weights_only=False)
pipeline.model.load_state_dict(chpt['model_state_dict'])
pipeline.model.eval()

In [ ]:
batch = next(iter(pipeline.val_loader))
inputs, targets = batch['image'], batch['label']

In [ ]:
import torch.nn.functional as F

In [ ]:
outputs = F.sigmoid(pipeline.model(inputs))

In [ ]:
plt.imshow(outputs[0, 0].detach().numpy()>0.75, cmap='gray')

In [ ]:
pipeline.metrics = SegmentationMetrics()
pipeline.topological_metrics = TopologicalMetrics(threshold=config['training'].get('metric_threshold', 0.5))

In [ ]:
pipeline.metrics.update(outputs.squeeze(1)>0.7, targets.squeeze(1))
pipeline.topological_metrics.update(outputs.squeeze(1)>0.75, targets.squeeze(1))

In [ ]:
pipeline.metrics.compute()

In [ ]:
pipeline.topological_metrics.compute()


## Section 5: Analyze Training Results

Plot training curves and compare convergence across different loss combinations.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Training Comparison Across Loss Configurations', fontsize=16, fontweight='bold')

# Loss curves
ax1 = axes[0, 0]
for loss_name, results in training_results.items():
    ax1.plot(results['train_loss'], label=f'{loss_name} (train)', alpha=0.7)
    ax1.plot(results['val_loss'], label=f'{loss_name} (val)', linestyle='--', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# Dice score curves
ax2 = axes[0, 1]
for loss_name, results in training_results.items():
    if 'f1' in results['val_metrics']:
        ax2.plot(results['val_metrics']['f1'], label=loss_name, marker='o', markersize=3)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Validation Dice Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

# IoU curves
ax3 = axes[1, 0]
for loss_name, results in training_results.items():
    if 'iou' in results['val_metrics']:
        ax3.plot(results['val_metrics']['iou'], label=loss_name, marker='s', markersize=3)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('IoU Score')
ax3.set_title('Validation IoU Score')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Final metrics comparison
ax4 = axes[1, 1]
metrics_data = []
for loss_name, results in validation_results.items():
    if 'metrics' in results:
        metrics = results['metrics']
        metrics_data.append({
            'config': loss_name,
            'dice': metrics.get('f1', 0),
            'iou': metrics.get('iou', 0),
            'accuracy': metrics.get('accuracy', 0),
            'precision': metrics.get('precision', 0),
            'recall': metrics.get('recall', 0)
        })

if metrics_data:
    df_metrics = pd.DataFrame(metrics_data)
    df_metrics.set_index('config').plot(kind='bar', ax=ax4, rot=45)
    ax4.set_title('Final Test Metrics')
    ax4.set_ylabel('Score')
    ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6: Detailed Metrics Analysis

Analyze final metrics including topological measures for each configuration.

In [ ]:
# Create comprehensive metrics table
print("=" * 100)
print("COMPREHENSIVE METRICS COMPARISON")
print("=" * 100)

# Collect all metrics
all_metrics = []
for loss_name, results in validation_results.items():
    if 'metrics' in results:
        metrics = results['metrics']
        row = {
            'Configuration': loss_name,
            'Test Loss': results.get('loss', 0),
            'Dice': metrics.get('f1', 0),
            'IoU': metrics.get('iou', 0),
            'Accuracy': metrics.get('accuracy', 0),
            'Precision': metrics.get('precision', 0),
            'Recall': metrics.get('recall', 0),
            'Betti_0_Pred': metrics.get('betti_pred_b0', 0),
            'Betti_1_Pred': metrics.get('betti_pred_b1', 0),
            'Betti_0_Target': metrics.get('betti_target_b0', 0),
            'Betti_1_Target': metrics.get('betti_target_b1', 0),
            'Connectivity_Match': metrics.get('connectivity_match', 0),
            'Skeleton_Accuracy': metrics.get('skeleton_accuracy', 0)
        }
        all_metrics.append(row)

# Create DataFrame
df_results = pd.DataFrame(all_metrics)
df_results = df_results.round(4)

# Display results
print("\nDetailed Results:")
print(df_results.to_string(index=False))

# Best performers
print("\n" + "=" * 100)
print("BEST PERFORMERS BY METRIC")
print("=" * 100)

metrics_to_rank = ['Dice', 'IoU', 'Accuracy', 'Precision', 'Recall', 'Connectivity_Match', 'Skeleton_Accuracy']
for metric in metrics_to_rank:
    if metric in df_results.columns:
        best_row = df_results.loc[df_results[metric].idxmax()]
        print("15")

## Section 7: Topological Analysis

Analyze how different loss combinations affect topological properties of vessel segmentation.

In [ ]:
# Topological analysis visualization
if all_metrics:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Topological Metrics Analysis', fontsize=16, fontweight='bold')
    
    # Betti numbers comparison
    ax1 = axes[0, 0]
    configs = df_results['Configuration']
    betti_0_pred = df_results['Betti_0_Pred']
    betti_0_target = df_results['Betti_0_Target']
    betti_1_pred = df_results['Betti_1_Pred']
    betti_1_target = df_results['Betti_1_Target']
    
    x = np.arange(len(configs))
    width = 0.2
    
    ax1.bar(x - width, betti_0_pred, width, label='Betti 0 (Pred)', alpha=0.7)
    ax1.bar(x, betti_0_target, width, label='Betti 0 (Target)', alpha=0.7)
    ax1.bar(x + width, betti_1_pred, width, label='Betti 1 (Pred)', alpha=0.7)
    ax1.bar(x + 2*width, betti_1_target, width, label='Betti 1 (Target)', alpha=0.7)
    
    ax1.set_xlabel('Configuration')
    ax1.set_ylabel('Betti Number')
    ax1.set_title('Betti Numbers: Predicted vs Target')
    ax1.set_xticks(x)
    ax1.set_xticklabels(configs, rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Connectivity and skeleton accuracy
    ax2 = axes[0, 1]
    connectivity = df_results['Connectivity_Match']
    skeleton_acc = df_results['Skeleton_Accuracy']
    
    ax2.bar(x - width/2, connectivity, width, label='Connectivity Match', alpha=0.7, color='skyblue')
    ax2.bar(x + width/2, skeleton_acc, width, label='Skeleton Accuracy', alpha=0.7, color='lightcoral')
    
    ax2.set_xlabel('Configuration')
    ax2.set_ylabel('Score')
    ax2.set_title('Connectivity & Skeleton Metrics')
    ax2.set_xticks(x)
    ax2.set_xticklabels(configs, rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Segmentation vs Topological trade-off
    ax3 = axes[0, 2]
    dice_scores = df_results['Dice']
    topo_scores = (df_results['Connectivity_Match'] + df_results['Skeleton_Accuracy']) / 2
    
    ax3.scatter(dice_scores, topo_scores, s=100, alpha=0.7)
    for i, config in enumerate(configs):
        ax3.annotate(config, (dice_scores[i], topo_scores[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    ax3.set_xlabel('Dice Score')
    ax3.set_ylabel('Average Topological Score')
    ax3.set_title('Segmentation vs Topological Performance')
    ax3.grid(True, alpha=0.3)
    
    # Loss components analysis
    ax4 = axes[1, 0]
    loss_values = df_results['Test Loss']
    ax4.bar(configs, loss_values, alpha=0.7, color='orange')
    ax4.set_xlabel('Configuration')
    ax4.set_ylabel('Test Loss')
    ax4.set_title('Final Test Loss by Configuration')
    ax4.set_xticklabels(configs, rotation=45)
    ax4.grid(True, alpha=0.3)
    
    # Performance ranking
    ax5 = axes[1, 1]
    # Create composite score
    df_results['Composite_Score'] = (
        df_results['Dice'] * 0.4 + 
        df_results['IoU'] * 0.3 + 
        df_results['Connectivity_Match'] * 0.2 + 
        df_results['Skeleton_Accuracy'] * 0.1
    )
    
    composite_scores = df_results['Composite_Score']
    sorted_indices = np.argsort(composite_scores)[::-1]
    sorted_configs = df_results['Configuration'].iloc[sorted_indices]
    sorted_scores = composite_scores.iloc[sorted_indices]
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(sorted_configs)))
    ax5.bar(range(len(sorted_configs)), sorted_scores, color=colors, alpha=0.7)
    ax5.set_xlabel('Rank')
    ax5.set_ylabel('Composite Score')
    ax5.set_title('Overall Performance Ranking')
    ax5.set_xticks(range(len(sorted_configs)))
    ax5.set_xticklabels([f'{i+1}. {config}' for i, config in enumerate(sorted_configs)], rotation=45)
    ax5.grid(True, alpha=0.3)
    
    # Remove empty subplot
    axes[1, 2].remove()
    
    plt.tight_layout()
    plt.show()
    
    # Print ranking summary
    print("\n" + "=" * 100)
    print("OVERALL PERFORMANCE RANKING")
    print("=" * 100)
    for i, (config, score) in enumerate(zip(sorted_configs, sorted_scores)):
        print("2d")

## Section 8: Save Results and Summary

Save the comparison results and provide final recommendations.

In [ ]:
# Save results to file
results_summary = {
    'timestamp': str(pd.Timestamp.now()),
    'loss_configs': loss_configs,
    'training_results': training_results,
    'validation_results': validation_results,
    'final_metrics': df_results.to_dict('records') if 'df_results' in locals() else all_metrics
}

with open('logs/combined_loss_comparison.json', 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

print("✓ Results saved to logs/combined_loss_comparison.json")

# Print final summary
print("\n" + "=" * 100)
print("EXPERIMENT SUMMARY")
print("=" * 100)

if all_metrics:
    best_overall = df_results.loc[df_results['Composite_Score'].idxmax()]['Configuration']
    best_dice = df_results.loc[df_results['Dice'].idxmax()]['Configuration']
    best_topo = df_results.loc[df_results['Connectivity_Match'].idxmax()]['Configuration']
    
    print(f"Best Overall Performance: {best_overall}")
    print(f"Best Dice Score: {best_dice}")
    print(f"Best Topological Preservation: {best_topo}")
    
    print("
Key Findings:")
    print(f"• Combined losses generally outperform single losses")
    print(f"• Topological losses help preserve vessel connectivity")
    print(f"• Trade-off exists between segmentation accuracy and topological preservation")
    print(f"• {best_overall} provides the best balance of all metrics")

print("
Next Steps:")
print("• Fine-tune weights for the best performing configuration")
print("• Test on additional datasets")
print("• Consider ensemble methods combining multiple loss functions")
print("• Investigate advanced topological losses (SATLoss, etc.)")

## Summary

This notebook demonstrated comprehensive comparison of different combined loss configurations for vessel segmentation:

### ✅ **Completed:**
- **Combined Loss Setup**: Defined weighted combinations of segmentation + topological losses
- **Training Comparison**: Trained models with 5 different loss configurations
- **Metrics Collection**: Computed segmentation (Dice, IoU) + topological (Betti, connectivity) metrics
- **Performance Analysis**: Ranked configurations by overall performance
- **Visualization**: Created comprehensive plots comparing training curves and final metrics

### 📊 **Key Results:**
- Combined losses (segmentation + topological) generally outperform single losses
- Topological losses improve vessel connectivity preservation
- Trade-off exists between pure segmentation accuracy and topological structure preservation
- Best configuration provides balanced performance across all metrics

### 🔧 **Technical Features:**
- **Flexible Loss Configuration**: Easy to define weighted combinations via YAML
- **Automatic Metric Collection**: Both standard and topological metrics computed automatically
- **Comprehensive Analysis**: Training curves, final metrics, and performance rankings
- **Result Persistence**: All results saved to JSON for further analysis

### 🎯 **Recommendations:**
1. Use the best-performing combined loss configuration for production training
2. Fine-tune loss weights based on specific dataset characteristics
3. Consider topological losses when vessel connectivity is critical
4. Monitor both segmentation and topological metrics during training

The pipeline now supports sophisticated loss combinations while maintaining clean, modular code architecture.